<a href="https://colab.research.google.com/github/Abishnavi17/Triangular_Arbitrage_detector/blob/main/Speech_Intern_Evaluation_ABISHNAVI_ME24B073.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gnani.ai — Speech Team Intern Evaluation

**Time budget: 60 minutes** (Problem 1: ~20 min · Problem 2: ~40 min)

**Instructions**
- Fill in every block marked `### YOUR CODE HERE`. Do not change function signatures, variable names, or the self-check cells — they depend on those staying as-is.
- Run cells **top to bottom**. Each task has a `# --- Self-check ---` cell right after it that prints `PASS` or raises an `AssertionError`.
- Problem 2 requires internet access to download a pretrained model (~150 MB) and a small audio dataset from the Hugging Face Hub. Run the setup cell first and let it finish before starting the fill-in-the-blank cells.
- You may use PyTorch / Hugging Face documentation. Please don't copy-paste a solution verbatim from an LLM/search — you'll be asked to walk through your reasoning afterward.
- The comment lines in the document would guide you throughout the test
- Save the notebook and submit it as instructed by your interviewer.

In [42]:
!pip install -q "transformers>=4.46" "datasets>=2.19" scikit-learn accelerate librosa

import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
from tqdm import tqdm

torch.manual_seed(0)
print("Setup OK. Torch version:", torch.__version__)

Setup OK. Torch version: 2.11.0+cpu


---
## Problem 1 — PyTorch Basics: Forward, Backward & Training Loop (≈20 min)

You're given a small synthetic 2D binary classification dataset (two separable clusters) and a partially-implemented model. Your job is to:
1. Finish the model's `forward` method.
2. Finish the training loop: forward pass → loss → zero the gradients → backpropagate → optimizer step.

This is the same loop structure used to train any PyTorch model, including the MLP classifier in Problem 2.

In [43]:
# --- Toy dataset: two separable 2D Gaussian blobs (given, do not modify) ---
n_per_class = 100
class0 = torch.randn(n_per_class, 2) * 0.6 + torch.tensor([-2.0, -2.0])
class1 = torch.randn(n_per_class, 2) * 0.6 + torch.tensor([2.0, 2.0])

X = torch.cat([class0, class1], dim=0)
y = torch.cat([torch.zeros(n_per_class), torch.ones(n_per_class)])

# shuffle
perm = torch.randperm(X.size(0))
X, y = X[perm], y[perm]

print("X shape:", X.shape, "y shape:", y.shape)

X shape: torch.Size([200, 2]) y shape: torch.Size([200])


### Task 1.1 — Model definition (forward pass)

Complete `forward`: pass `x` through `fc1`, apply a ReLU, then pass through `fc2`. The output should be a single raw logit per example (no sigmoid here — the loss function will apply it).

In [44]:
#f.relu
class BinaryClassifier(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=16):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        ### YOUR CODE HERE
        # TODO: fc1 -> ReLU -> fc2, return the raw logit (shape: [batch, 1])
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x

model = BinaryClassifier()
test_out = model(X[:5])
print("Test output shape:", None if test_out is None else test_out.shape)


Test output shape: torch.Size([5, 1])


In [45]:
# --- Self-check ---
assert test_out is not None, "forward() returned None"
assert test_out.shape == (5, 1), f"Expected shape (5, 1), got {tuple(test_out.shape)}"
print("PASS: Task 1.1")

PASS: Task 1.1


### Task 1.2 — Training loop

`criterion` and `optimizer` are given. Complete the training loop body:
1. Run the forward pass to get `logits`.
2. Compute `loss` with `criterion` against `y`.
3. Zero out old gradients.
4. Backpropagate.
5. Take an optimizer step.

Record each epoch's loss into `loss_history` (already set up) so the self-check can verify training actually worked.

In [46]:
model = BinaryClassifier()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

num_epochs = 60
loss_history = []

for epoch in tqdm(range(num_epochs)):
    ### YOUR CODE HERE
    # TODO 1: forward pass -> logits (shape [N, 1])
    logits = model(X)

    # TODO 2: compute the loss (criterion expects matching shapes: squeeze logits to [N])
    loss =  criterion(logits.squeeze(-1), y)

    # TODO 3: zero the gradients
    optimizer.zero_grad()


    # TODO 4: backpropagate
    loss.backward()

    # TODO 5: optimizer step
    optimizer.step()


    loss_history.append(loss.item() if loss is not None else float("nan"))

print("First epoch loss:", loss_history[0])
print("Last epoch loss:", loss_history[-1])


100%|██████████| 60/60 [00:00<00:00, 938.37it/s]

First epoch loss: 0.779592752456665
Last epoch loss: 1.063044874172192e-05


In [47]:
# --- Self-check ---
assert len(loss_history) == num_epochs, "loss_history wasn't populated for every epoch"
assert not any(np.isnan(l) for l in loss_history), "Got NaN loss — check your forward/loss wiring"
assert loss_history[-1] < loss_history[0] * 0.5, (
    f"Loss should drop substantially over {num_epochs} epochs "
    f"(first={loss_history[0]:.4f}, last={loss_history[-1]:.4f}) — check zero_grad/backward/step ordering"
)

with torch.no_grad():
    preds = (torch.sigmoid(model(X)) > 0.5).float().squeeze(-1)
    accuracy = (preds == y).float().mean().item()
print(f"Final train accuracy: {accuracy:.3f}")
assert accuracy > 0.9, f"Expected >0.9 accuracy on this easily-separable toy data, got {accuracy:.3f}"
print("PASS: Task 1.2")

Final train accuracy: 1.000
PASS: Task 1.2


---
## Problem 2 — Language Identification with Whisper Tiny (≈40 min)

You'll build a simple Language Identification (LID) system:
1. Load audio samples in 3 languages (English, Hindi, Tamil) from Mozilla Common Voice.
2. Extract fixed-length feature vectors from each audio clip using Whisper Tiny's encoder.
3. Train a 2-layer MLP classifier (same pattern as Problem 1!) to predict the language from the encoder features.

This mirrors a real feature-extraction workflow: use a pretrained model as a feature extractor, then train a lightweight classifier on top.

### Setup — Load audio samples (given, do not modify)

We load 60 audio clips (20 per language: English, Hindi, Tamil) from the `gnani/lid-data-train-sample` dataset. The audio is resampled to 16 kHz (Whisper's expected input rate).

In [48]:
import os

print(os.listdir("/content"))

['.config', 'lid.tar', 'lid', 'sample_data']


In [49]:
import tarfile

with tarfile.open("/content/lid.tar", "r") as tar:
    tar.extractall("/content")

print("Dataset extracted")

/tmp/ipykernel_1021/3078464558.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("/content")


Dataset extracted


In [50]:
import os

for root, dirs, files in os.walk("/content"):
    if files:
        print(root, "→", len(files), "files")

/content → 1 files
/content/.config → 8 files
/content/.config/configurations → 1 files
/content/.config/logs/2026.09.04 → 6 files
/content/lid → 1 files
/content/lid/tamil → 20 files
/content/lid/english → 20 files
/content/lid/.cache/huggingface → 1 files
/content/lid/.cache/huggingface/download → 2 files
/content/lid/.cache/huggingface/download/tamil → 40 files
/content/lid/.cache/huggingface/download/english → 40 files
/content/lid/.cache/huggingface/download/hindi → 40 files
/content/lid/hindi → 20 files
/content/sample_data → 6 files


In [51]:
import glob
import librosa


audio_data = []
labels = []

label_map = {
    "english": 0,
    "hindi": 1,
    "tamil": 2
}

for language,label in label_map.items():
  files=glob.glob(f"/content/lid/{language}/*.wav")

  for file in files:
    audio,sr=librosa.load(file,sr=16000)

    audio_data.append(audio)
    labels.append(label)

print(f"Loaded {len(audio_data)} audio samples")
print(f"Labels distribution: {{l: labels.count(l) for l in sorted(set(labels))}}")

Loaded 60 audio samples
Labels distribution: {l: labels.count(l) for l in sorted(set(labels))}


### Task 2.1 — Load Whisper Tiny processor & model (5 min)

Load the `WhisperProcessor` and `WhisperModel` (the base encoder-decoder model, **not** `WhisperForConditionalGeneration`) for `openai/whisper-tiny`. Set the model to evaluation mode so dropout is disabled.

We use the base model because we only need the encoder to extract features — we won't be generating text.

In [52]:
from transformers import WhisperProcessor, WhisperModel

MODEL_NAME = "openai/whisper-tiny"

### YOUR CODE HERE
# TODO: Load the WhisperProcessor for MODEL_NAME
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")

# TODO: Load the WhisperModel for MODEL_NAME
whisper_model = WhisperModel.from_pretrained("openai/whisper-tiny")
# TODO: Set whisper_model to evaluation mode
whisper_model.eval()

print("Processor loaded:", processor is not None)
print("Model loaded:", whisper_model is not None)
print("Encoder hidden size:", whisper_model.config.d_model if whisper_model is not None else None)


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Processor loaded: True
Model loaded: True
Encoder hidden size: 384


In [53]:
# --- Self-check ---
assert processor is not None, "processor is None"
assert whisper_model is not None, "whisper_model is None"
assert hasattr(whisper_model, "encoder"), "Model should have an 'encoder' attribute — did you load WhisperModel (not WhisperForConditionalGeneration)?"
assert whisper_model.config.d_model == 384, f"Expected Whisper Tiny hidden size 384, got {whisper_model.config.d_model}"
assert not whisper_model.training, "Model should be in eval mode (call .eval())"
print("PASS: Task 2.1")

PASS: Task 2.1


### Task 2.2 — Extract encoder features (12 min)

For each audio sample:
1. Use `processor` to convert the raw audio waveform into mel-spectrogram input features. Pass `sampling_rate=16000` and `return_tensors="pt"`.
2. Feed `input_features` through `whisper_model.encoder` to get the encoder output.
3. Mean-pool `encoder_output.last_hidden_state` across the time dimension (dim=1) to get a single fixed-length 384-dim vector per audio clip.
4. Append the resulting vector to a `features` list.

After the loop, stack all features into a single tensor `X_audio` of shape `[60, 384]`.

In [54]:
features = []

with torch.no_grad():
    for audio in tqdm(audio_data):
        ### YOUR CODE HERE
        # TODO 1: Use processor to convert audio to input features
        # processor(audio, sampling_rate=16000, return_tensors="pt")
        inputs = processor(audio,sampling_rate=16000,return_tensors="pt")


        # TODO 2: Pass inputs.input_features through whisper_model.encoder
        encoder_output = whisper_model.encoder(inputs.input_features)

        # TODO 3: Get last_hidden_state from encoder_output -> shape [1, T, 384]
        hidden = encoder_output.last_hidden_state

        # TODO 4: Mean-pool over the time dimension (dim=1) and squeeze -> shape [384]
        pooled = hidden.mean(dim=1).squeeze(0)

        # TODO 5: Append pooled to features list
        features.append(pooled)

X_audio = torch.stack(features) if len(features) > 0 else torch.zeros(45, 384)
print("Feature matrix shape:", X_audio.shape)


100%|██████████| 60/60 [01:50<00:00,  1.85s/it]

Feature matrix shape: torch.Size([60, 384])


In [55]:
# --- Self-check ---
assert X_audio.shape == (60, 384), f"Expected shape (60, 384), got {tuple(X_audio.shape)}"
assert not torch.isnan(X_audio).any(), "Feature matrix contains NaN values"
assert not (X_audio == 0).all(), "Feature matrix is all zeros — encoder output looks wrong"
print("PASS: Task 2.2")

PASS: Task 2.2


### Task 2.3 — Define MLP classifier (8 min)

Define a 2-layer MLP for 3-class language classification:
- `fc1`: Linear(384 → 128)
- ReLU activation
- `fc2`: Linear(128 → 3)

The output should be 3 raw logits (one per language). This is the same pattern as Problem 1's `BinaryClassifier`, just with different input/output dimensions.

In [56]:
#f.relu

class LIDClassifier(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=128, num_classes=3):
        super().__init__()
        ### YOUR CODE HERE
        # TODO: Define self.fc1 (Linear: input_dim -> hidden_dim)
        # TODO: Define self.fc2 (Linear: hidden_dim -> num_classes)
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):

        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)

        return x

lid_model = LIDClassifier()
test_out = lid_model(X_audio[:5])
print("Test output shape:", None if test_out is None else test_out.shape)


Test output shape: torch.Size([5, 3])


In [57]:
# --- Self-check ---
assert test_out is not None, "forward() returned None"
assert test_out.shape == (5, 3), f"Expected shape (5, 3), got {tuple(test_out.shape)}"
print("PASS: Task 2.3")

PASS: Task 2.3


### Task 2.4 — Train the LID classifier (15 min)

Train the MLP using `CrossEntropyLoss` (the multi-class equivalent of `BCEWithLogitsLoss` from Problem 1). The training loop is the same 5-step pattern:
1. Forward pass → logits
2. Compute loss (`CrossEntropyLoss` expects logits of shape `[N, C]` and integer labels of shape `[N]`)
3. Zero gradients
4. Backpropagate
5. Optimizer step

We split the 60 samples into 45 train / 15 test (stratified: 5 test per language).

In [58]:
# --- Given: encode labels and create train/test split (do not modify) ---
from sklearn.model_selection import train_test_split

label_map = {"en": 0, "hi": 1, "ta": 2}
y_audio = torch.tensor([l for l in labels])

indices = list(range(len(y_audio)))
train_idx, test_idx = train_test_split(indices, test_size=15, stratify=y_audio.numpy(), random_state=42)

X_train, X_test = X_audio[train_idx], X_audio[test_idx]
y_train, y_test = y_audio[train_idx], y_audio[test_idx]
print(f"Train: {len(X_train)} samples, Test: {len(X_test)} samples")
print(f"Test labels: {y_test.tolist()} (0=en, 1=hi, 2=ta)")

Train: 45 samples, Test: 15 samples
Test labels: [0, 2, 1, 0, 1, 2, 2, 0, 1, 2, 1, 1, 0, 2, 0] (0=en, 1=hi, 2=ta)


In [59]:

lid_model = LIDClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lid_model.parameters(), lr=0.01)

num_epochs = 100
loss_history = []

for epoch in tqdm(range(num_epochs)):
    ### YOUR CODE HERE

    # TODO 1: forward pass -> logits = lid_model(X_train)
    logits = lid_model(X_train)
    # TODO 2: compute loss = criterion(logits, y_train)
    loss = criterion(logits,y_train)

    # TODO 3: zero the gradients
    optimizer.zero_grad()

    # TODO 4: backpropagate

    loss.backward()

    # TODO 5: optimizer step
    optimizer.step()

    loss_history.append(loss.item() if loss is not None else float("nan"))

print(f"First epoch loss: {loss_history[0]:.4f}")
print(f"Last epoch loss: {loss_history[-1]:.4f}")


100%|██████████| 100/100 [00:00<00:00, 441.12it/s]

First epoch loss: 1.1486
Last epoch loss: 0.0018


In [60]:
# --- Self-check ---
assert len(loss_history) == num_epochs, "loss_history wasn't populated for every epoch"
assert not any(np.isnan(l) for l in loss_history), "Got NaN loss — check your forward/loss wiring"
assert loss_history[-1] < loss_history[0] * 0.5, (
    f"Loss should drop substantially over {num_epochs} epochs "
    f"(first={loss_history[0]:.4f}, last={loss_history[-1]:.4f}) — check zero_grad/backward/step ordering"
)
print("PASS: Task 2.4 (training)")

PASS: Task 2.4 (training)


### Evaluation (given — just execute)

In [61]:
lid_model.eval()
with torch.no_grad():
    test_logits = lid_model(X_test)
    preds = test_logits.argmax(dim=1)
    accuracy = (preds == y_test).float().mean().item()

inv_label_map = {v: k for k, v in label_map.items()}
print(f"Test accuracy: {accuracy:.3f} ({int(accuracy * len(y_test))}/{len(y_test)} correct)")
for i in range(len(y_test)):
    status = "✓" if preds[i] == y_test[i] else "✗"
    print(f"  {status} True: {inv_label_map[y_test[i].item()]} | Pred: {inv_label_map[preds[i].item()]}")

Test accuracy: 0.800 (12/15 correct)
  ✗ True: en | Pred: hi
  ✓ True: ta | Pred: ta
  ✓ True: hi | Pred: hi
  ✓ True: en | Pred: en
  ✓ True: hi | Pred: hi
  ✓ True: ta | Pred: ta
  ✗ True: ta | Pred: hi
  ✓ True: en | Pred: en
  ✓ True: hi | Pred: hi
  ✓ True: ta | Pred: ta
  ✓ True: hi | Pred: hi
  ✗ True: hi | Pred: ta
  ✓ True: en | Pred: en
  ✓ True: ta | Pred: ta
  ✓ True: en | Pred: en


In [62]:
# --- Self-check ---
assert accuracy > 0.5, (
    f"Expected better than random (>0.5) for 3-class LID with Whisper features, got {accuracy:.3f} "
    "— check your feature extraction and training loop"
)
print(f"PASS: Task 2.4 (evaluation) — Test accuracy: {accuracy:.3f}")

PASS: Task 2.4 (evaluation) — Test accuracy: 0.800


---
## Short-Answer Reasoning (≈5 min)

Answer directly in the cell below. There's no single correct answer — we're evaluating how you reason.

In [63]:
answer_c1 = """
Q1: In Problem 1, what would happen if you forgot to call optimizer.zero_grad() before loss.backward()?
Why does PyTorch require you to zero gradients manually instead of doing it automatically each step?

YOUR ANSWER:If zero_grad() is not called, gradients accumulate from previous iterations, causing incorrect parameter updates.
PyTorch requires manual zeroing because gradient accumulation is useful for techniques like gradient accumulation across multiple batches.

"""

answer_c2 = """
Q2: How would you improve LID performance in case of overlapping languages like Tamil and Malayalam or Hindi and Marathi?

YOUR ANSWER:I would use more diverse samples and fine-tune the model on language-identification data.
This can help the model learn subtle language-specific features and reduce confusion between similar languages.

"""

answer_c3 = """
Q3: With only 15 samples per language, what are two concrete things you'd try to improve LID accuracy
(besides collecting more data), and why might each help?

YOUR ANSWER:
1. Data augmentation: add noise or vary speed to make the model more robust to different speaking conditions.
2. Fine-tuning: fine-tune the pretrained Whisper features for LID so the model learns features specific to the target languages.

"""

print("Answers recorded.")

Answers recorded.


---
### Done!

Save the notebook and submit it as instructed. Be ready to walk through your reasoning for the training loop (Problem 1) and the feature extraction + classifier (Problem 2).